# Compute daily mean wind speed

To align with DCPP model data, we will compute daily wind speed from daily mean u and v components.

- ERA5
- BARRA-R2
- BARRA-C2 later - not a reanalysis as no DA

In [80]:
client.close()
cluster.close()

INFO:dask_jobqueue.pbs:Resource specification for PBS not set, initializing it to select=1:ncpus=24:mem=90GB


In [1]:
from dask.distributed import Client,LocalCluster
from dask_jobqueue import PBSCluster

In [2]:
PROJECT = "dt6"

In [3]:
walltime = "01:00:00"
cores = 24
memory = str(4 * cores) + "GB"

cluster = PBSCluster(
    walltime=str(walltime),
    cores=cores,
    memory=str(memory),
    processes=cores,
    job_extra_directives=[
        "-q normal",
        "-P "+PROJECT,
        "-l ncpus="+str(cores),
        "-l mem="+str(memory),
        "-l storage=gdata/xp65+gdata/ob53+gdata/w42+scratch/w42+gdata/gb02+scratch/gb02+gdata/ng72+scratch/ng72+gdata/rt52"
    ],
    local_directory="$TMPDIR",
    job_directives_skip=["select"],
    log_directory="/scratch/w42/dr6273/tmp/logs"
)

In [4]:
cluster.scale(jobs=1)
client = Client(cluster)

In [5]:
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://10.6.121.4:36053,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [6]:
import xarray as xr

import glob

In [7]:
%cd /g/data/w42/dr6273/work/wind_drought/
import functions as fn

%load_ext autoreload
%autoreload 2

/g/data/w42/dr6273/work/wind_drought


In [8]:
BARRA_R2_PATH = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/" # daily mean (?)

WRITE_PATH = "/g/data/ng72/dr6273/work/projects/wind_drought/data/BARRA-R2/"

# Compute wind speed by year

In [9]:
def barraR2_preprocess(ds):
    """
    Preprocess function for open_mfdataset.
    Selects Australian region, computes daily mean and renames coords.
    """
    ds = ds.sel(
        lon=slice(REGION[0], REGION[1]),
        lat=slice(REGION[3], REGION[2])
    )
    # ds["time"] = ds["time"].dt.floor("D")
    # ds = ds.chunk({"time": -1, "lat": -1, "lon": -1})
    # ds = ds.astype("float32")
    return ds

In [10]:
# def load_daily_barraR2(preprocess, variable, year, data_path=BARRA_R2_PATH):
#     """
#     Load and preprocess hourly data for a given year
    
#     preprocess: preprocess function
#     variable: name of variable to process
#     year: year to process
#     data_path: path to hourly data
#     """
#     # Open all hours in the year (~33 GB)
#     da = xr.open_mfdataset(
#         data_path + variable + "/latest/*.nc",
#         preprocess=preprocess
#     )
#     return da

In [11]:
def open_barra(files):
    """
    Open multiple files and preprocess to region.
    
    fp: str, path to file. Should not include files, only the path to dir.
    lat_slice, lon_slice: slice of lat/lon to subset
    lat_name, lon_name: names of lat/lon coords.
    """
    # def preprocess(ds):
    #     # ds = ds.rename({lat_name: "lat"})
    #     # ds = ds.rename({lon_name: "lon"})
    #     ds = ds.sel(lon=lon_slice, lat=lat_slice)
    #     # ds["time"] = ds["time"].dt.round(freq="20min")
    #     return ds.astype("float32")
    
    ds = xr.open_mfdataset(
        files,
        preprocess=barraR2_preprocess,
        chunks={"time": "300MB"},# "lat": -1, "lon": -1},
        concat_dim="time",
        combine="nested",
        compat="override",
        coords="minimal",
        data_vars="minimal",
        # decode_times=True
        # parallel=True
    )
    return ds

In [12]:
# def get_filename(variable, frequency, year, month):
#     """
#     Return BARRA-R2 AUS-11 historical data filepaths
    
#     variable: str, variable name e.g. ua100m
#     frequency: str, timestep e.g. 20min, 1hr
#     year: str
#     month: str, month in format '01', '02', ..., '12'
#     """
#     file_template = variable + "_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1_" + frequency + "_"
#     return barra_path + frequency + "/" + variable + "/latest/" + file_template + year + month + "-" + year + month + ".nc"

In [13]:
REGION = fn.get_Aus_boundary()

In [14]:
YEARS = range(1979, 2025)

In [15]:
def barraR2_windspeed(u_name, v_name, ws_name, years, write_path):
    """
    Compute wind speed from daily mean u and v winds.

    u_name, v_name: str, names of u and v variables in BARRA-R2
    ws_name: str, desired wind speed variable name
    years: range, years to process
    write_path: str, path to write directory
    """
    da_list = []
    for year in years:
        if year % 5 == 0:
            print(year)
            
        # monthly_arrays = []
        # for month in month_str[:]:
            
        u = open_barra(
            sorted(glob.glob(BARRA_R2_PATH + u_name + "/latest/*_" + str(year) + "*.nc")),
        ).astype("float32")
        
        v = open_barra(
            sorted(glob.glob(BARRA_R2_PATH + v_name + "/latest/*_" + str(year) + "*.nc")),
        ).astype("float32")
        
        ws = fn.windspeed(
            u.rename({u_name: ws_name}),
            v.rename({v_name: ws_name})
        )[ws_name]

        ws["time"] = ws["time"].dt.floor("D")

        # Time chunk size of 1 as it is the greatest common divisor of
        #. the different time steps associated with each year (365 and 366)
        # ws = ws.chunk({"time": 1, "lat": -1, "lon": -1})
        
        ws = ws.to_dataset(name=ws_name)
        
        encoding = {
            ws_name: {"dtype": "float32"}
        }

        ws.to_netcdf(
            write_path + ws_name + "_BARRA-R2_daily_Aus_"+str(year)+".nc",
            mode="w"
        )
        
        # if year == 1979:
        #     ws.to_zarr(
        #         write_path + ws_name + "_BARRA-R2_daily_Aus.zarr",
        #         mode="w",
        #         encoding=encoding,
        #         consolidated=True,
        #         zarr_format=2, # Need this to prevent an error with zarr3: https://github.com/pydata/xarray/issues/10032
        #     )
        # else:
        #     ws.to_zarr(
        #         write_path + ws_name + "_BARRA-R2_daily_Aus.zarr",
        #         mode="a",
        #         append_dim="time",
        #         consolidated=True,
        #         zarr_format=2, # Need this to prevent an error with zarr3: https://github.com/pydata/xarray/issues/10032
        #     )

        # return ws

        # da_list.append(ws)
        
    # ws = xr.concat(da_list, dim="time")

    # encoding = {
    #     ws_name: {"dtype": "float32"}
    # }

    # ws = ws.chunk({"time": 550, "lat": -1, "lon": -1})
    # ws = ws.to_dataset(name=ws_name)
    # ws.to_zarr(
    #     write_path + ws_name + "_BARRA-R2_daily_Aus.zarr",
    #     consolidated=True,
    #     mode="w",
    #     zarr_format=2, # Need this to prevent an error with zarr3: https://github.com/pydata/xarray/issues/10032
    #     encoding=encoding
    # )
    # # return ws

In [13]:
# barraR2_windspeed("ua100m", "va100m", "ws100m", YEARS[3:], WRITE_PATH)

In [77]:
BARRA_R2_PATH

'/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/'

In [18]:
YEARS[10:]

range(1989, 2025)

In [19]:
barraR2_windspeed("uas", "vas", "wss", YEARS[10:], WRITE_PATH)

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 26.11 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


1990


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 26.11 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 25.48 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


1995
2000
2005


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 13.25 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


2010
2015
2020


# Close cluster

In [21]:
client.close()
cluster.close()